# 02 · Preprocessing

Build the window dataset that both the RF baseline and the CNN will share. Pipeline: load each subject, rectify (skip bandpass — see DB1 caveat in `src/preprocess.py`), window at 200 ms / 100 ms overlap, save to `data/processed/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
from tqdm import tqdm

from src.data import DB1_SUBJECTS, load_or_synthesize
from src.preprocess import rectify, window_signal

DATA_DIR = ROOT / 'data' / 'raw'
OUT_DIR = ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# For a fast smoke test, set SUBJECTS = [1, 2, 3]. Use DB1_SUBJECTS for the full run.
SUBJECTS = [1, 2, 3]
WINDOW_MS = 200
OVERLAP_MS = 100

In [ ]:
all_windows, all_labels, all_subjects, all_reps = [], [], [], []
for s in tqdm(SUBJECTS):
    rec = load_or_synthesize(subject=s, data_dir=DATA_DIR, seed=s)
    emg = rectify(rec.emg)
    out = window_signal(emg, rec.stimulus, rec.repetition, fs=rec.fs,
                        window_ms=WINDOW_MS, overlap_ms=OVERLAP_MS)
    all_windows.append(out['windows'])
    all_labels.append(out['labels'])
    all_reps.append(out['reps'])
    all_subjects.append(np.full(out['labels'].shape[0], s, dtype=np.int64))

windows = np.concatenate(all_windows)
labels = np.concatenate(all_labels)
subjects = np.concatenate(all_subjects)
reps = np.concatenate(all_reps)
print('windows:', windows.shape, 'labels:', labels.shape, 'classes:', np.unique(labels).size)

In [ ]:
np.savez_compressed(
    OUT_DIR / 'windows.npz',
    windows=windows, labels=labels, subjects=subjects, reps=reps,
)
print('saved', OUT_DIR / 'windows.npz')